# Detecção de Anomalias em Transações

**O que será abordado:**
1.  **Carregamento e Exploração de Dados:** Iniciaremos com a importação de um dataset base.
2.  **Simulação de Anomalias e Feature Engineering:** Criaremos um cenário artificial de anomalias e desenvolveremos novas características para enriquecer o conjunto de dados.
3.  **Regressão Logística em Dados Desbalanceados:** Demonstraremos como modelos simples podem falhar em dados desbalanceados.
4.  **Balanceamento de Dados (SMOTE):** Aplicaremos SMOTE para mitigar o problema do desbalanceamento.
5.  **Modelagem Avançada (XGBoost) e Explicabilidade (SHAP):** Utilizaremos um modelo robusto e ferramentas para entender suas decisões.


In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"

df = pd.read_csv(url)

df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4



## 2. Simulação de Anomalias e Feature Engineering

Com os dados brutos carregados, o próximo passo crucial é preparar o conjunto de dados para a detecção de anomalias. Como o dataset original não contém anomalias reais para o nosso propósito, iremos **simular um cenário de anomalias artificiais** e, em seguida, realizar **engenharia de features** para criar novas variáveis que possam ser úteis para o modelo de detecção. Esta etapa é fundamental para transformar os dados transacionais em um formato adequado para análise e modelagem de anomalias.

In [ ]:
import numpy as np

# Criando o cenário artificial de anomalia (ex: contas muito altas ou muitas pessoas)
# Como a base original não tem fraude, criamos uma regra para simular o desbalanceamento (~5% de anomalias)
df['is_anomaly'] = np.where((df['total_bill'] > 40) | (df['tip'] > 7), 1, 0)

# Feature Engineering (Criando novas métricas explicativas)
df['pct_gorjeta'] = df['tip'] / df['total_bill']
df['gasto_por_pessoa'] = df['total_bill'] / df['size']

# Tratando as variáveis categóricas (One-Hot Encoding)
# Transforma sex, smoker, day e time em colunas numéricas (0 ou 1)
df_final = pd.get_dummies(df, columns=['sex', 'smoker', 'day', 'time'], drop_first=True)

print("Estrutura do df_final criada com sucesso!")
print(f"Total de registros: {len(df_final)} | Anomalias encontradas: {df_final['is_anomaly'].sum()}")

## 4. Regressão Logística para Detecção de Anomalias (Dados Desbalanceados)

Nesta seção, aplicaremos um modelo de Regressão Logística para classificar transações como anomalias ou não anomalias. Nosso objetivo principal é demonstrar como a métrica de acurácia pode ser enganosa em conjuntos de dados desbalanceados e a importância de métricas como Recall, Precisão e F1-Score.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Definir features (X) e variável alvo (y)
# df_final contém as features engenheiradas e codificadas, e a coluna 'is_anomaly'

X = df_final.drop(columns=['is_anomaly'])
y = df_final['is_anomaly']

# Dividir os dados em conjuntos de treinamento e teste
# Usamos stratify=y para garantir que a proporção das classes seja mantida nos conjuntos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Distribuição da classe 'is_anomaly' no conjunto de treinamento:")
print(y_train.value_counts())
print("\nDistribuição da classe 'is_anomaly' no conjunto de teste:")
print(y_test.value_counts())


### Treinando o Modelo com Dados Desbalanceados

Vamos agora treinar um modelo de Regressão Logística e avaliar seu desempenho no conjunto de teste. Observaremos as métricas para a classe minoritária (anomalia) para entender o impacto do desbalanceamento.

In [ ]:
# Inicializar e treinar o modelo de Regressão Logística
model_lr = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' é bom para datasets pequenos
model_lr.fit(X_train, y_train)

# Fazer previsões no conjunto de teste
y_pred_lr = model_lr.predict(X_test)

print("--- Relatório de Classificação ---")
print(classification_report(y_test, y_pred_lr))

print("\n--- Matriz de Confusão ---")
cm = confusion_matrix(y_test, y_pred_lr)
print(cm)

# Visualizar a Matriz de Confusão
fig, ax = plt.subplots(figsize=(6, 6))
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não Anomalia', 'Anomalia'])
cmd.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Matriz de Confusão (Regressão Logística)')
plt.show()


#### Análise dos Resultados:

Como você pode observar no `classification_report` e na `confusion_matrix`:

*   **Acurácia (Accuracy):** Provavelmente alta (por exemplo, 95% ou mais). Isso ocorre porque o modelo está correto na maioria das vezes para a classe majoritária ('Não Anomalia').
*   **Recall para a classe '1' (Anomalia):** Muito baixo, possivelmente 0.00. Isso significa que o modelo falhou em identificar a maioria ou todas as anomalias reais. Ele está classificando quase tudo como 'Não Anomalia'.
*   **Precisão (Precision) para a classe '1' (Anomalia):** Se o recall for 0, a precisão também será 0 (ou indefinida). Se ele previu algumas anomalias, a precisão indicará a proporção de previsões de anomalia que estavam corretas.
*   **F1-Score para a classe '1' (Anomalia):** Será baixo ou zero, pois é a média harmônica de precisão e recall.

Este é o comportamento clássico de um modelo treinado em dados desbalanceados sem tratamento adequado: ele otimiza a acurácia geral em detrimento da detecção da classe minoritária, que é a mais importante neste cenário de anomalias.

## 5. Balanceamento de Dados (Oversampling com SMOTE)

Como vimos na seção anterior, a Regressão Logística em dados desbalanceados falhou em detectar a maioria das anomalias. Para mitigar isso, aplicaremos técnicas de balanceamento de dados. Usaremos o **SMOTE (Synthetic Minority Over-sampling Technique)** para gerar amostras sintéticas da classe minoritária, balanceando assim o conjunto de treinamento.

In [ ]:
from imblearn.over_sampling import SMOTE

print("Distribuição da classe 'is_anomaly' antes do SMOTE no treinamento:")
print(y_train.value_counts())

# Aplicar SMOTE ao conjunto de treinamento
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nDistribuição da classe 'is_anomaly' após o SMOTE no treinamento:")
print(y_train_res.value_counts())

# Treinar um novo modelo de Regressão Logística nos dados balanceados
model_lr_balanced = LogisticRegression(random_state=42, solver='liblinear')
model_lr_balanced.fit(X_train_res, y_train_res)

# Fazer previsões no conjunto de teste (mantemos o teste original para avaliação realista)
y_pred_lr_balanced = model_lr_balanced.predict(X_test)

print("\n--- Relatório de Classificação (Após SMOTE) ---")
print(classification_report(y_test, y_pred_lr_balanced))

print("\n--- Matriz de Confusão (Após SMOTE) ---")
cm_balanced = confusion_matrix(y_test, y_pred_lr_balanced)
print(cm_balanced)

# Visualizar a Matriz de Confusão após SMOTE
fig, ax = plt.subplots(figsize=(6, 6))
cmd_balanced = ConfusionMatrixDisplay(confusion_matrix=cm_balanced, display_labels=['Não Anomalia', 'Anomalia'])
cmd_balanced.plot(cmap=plt.cm.Greens, ax=ax)
plt.title('Matriz de Confusão (Regressão Logística com SMOTE)')
plt.show()

#### Análise dos Resultados Após Balanceamento:

Com o SMOTE, você deve observar uma melhoria notável no **Recall para a classe '1' (Anomalia)**. Isso indica que o modelo agora é mais capaz de identificar as anomalias reais. No entanto, é possível que a Precisão para a classe '1' tenha diminuído um pouco, pois o modelo pode estar fazendo mais previsões positivas (anomalias), incluindo alguns falsos positivos. O F1-Score é uma boa métrica para avaliar o equilíbrio entre Precisão e Recall.

Este resultado demonstra a importância de técnicas de balanceamento de dados para garantir que modelos em problemas de classificação desbalanceada possam efetivamente detectar a classe minoritária.

## 6. Modelo Avançado - XGBoost, Importância de Variáveis e Explicabilidade (SHAP)

Agora, vamos aplicar um modelo mais poderoso, o **XGBoost**, que é conhecido por seu alto desempenho em uma variedade de problemas de machine learning. Também exploraremos a importância das variáveis e a explicabilidade do modelo usando SHAP.

In [ ]:
# Instalar a biblioteca XGBoost e shap, se necessário.
# !pip install xgboost shap

import xgboost as xgb
import shap

# O XGBoost pode ser sensível ao balanceamento, então usaremos os dados balanceados do SMOTE
# ou podemos ajustar o parâmetro scale_pos_weight
# Para simplicidade, vamos usar os dados balanceados pelo SMOTE para o treinamento inicial.

# Inicializar e treinar o modelo XGBoost
# Usamos `scale_pos_weight` para lidar com o desbalanceamento diretamente no XGBoost
# Neste caso, como já usamos SMOTE, podemos manter o peso padrão ou ajustá-lo.
# Calculando scale_pos_weight para o treinamento balanceado:
# count_neg / count_pos
count_neg = y_train_res.value_counts()[0]
count_pos = y_train_res.value_counts()[1]
scale_pos_weight_value = count_neg / count_pos

model_xgb = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_estimators=100,
    learning_rate=0.1,
    # scale_pos_weight=scale_pos_weight_value # Descomente para usar o peso calculado
)

model_xgb.fit(X_train_res, y_train_res)

# Fazer previsões no conjunto de teste
y_pred_xgb = model_xgb.predict(X_test)

print("--- Relatório de Classificação (XGBoost) ---")
print(classification_report(y_test, y_pred_xgb))

print("\n--- Matriz de Confusão (XGBoost) ---")
cm_xgb = confusion_matrix(y_test, y_pred_xgb)
print(cm_xgb)

# Visualizar a Matriz de Confusão do XGBoost
fig, ax = plt.subplots(figsize=(6, 6))
cmd_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_xgb, display_labels=['Não Anomalia', 'Anomalia'])
cmd_xgb.plot(cmap=plt.cm.Purples, ax=ax)
plt.title('Matriz de Confusão (XGBoost)')
plt.show()


### Importância das Variáveis

O XGBoost oferece uma maneira fácil de obter a importância das variáveis, que indica quais features foram mais impactantes na tomada de decisão do modelo.

In [ ]:
print("Importância das Variáveis (XGBoost):")
feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': model_xgb.feature_importances_})
feature_importances = feature_importances.sort_values(by='importance', ascending=False)
display(feature_importances)

# Visualizar a importância das variáveis
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importances, palette='viridis')
plt.title('Importância das Variáveis no Modelo XGBoost')
plt.xlabel('Importância')
plt.ylabel('Variável')
plt.tight_layout()
plt.show()

### Ajuste de Hiperparâmetros (Grid Search Simplificado)

Para otimizar ainda mais o modelo XGBoost, podemos ajustar seus hiperparâmetros. Isso geralmente é feito com técnicas como Grid Search ou Random Search. Para manter a demonstração concisa, faremos um ajuste simplificado.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Definir um dicionário de hiperparâmetros para testar
# Para uma busca mais completa, adicione mais valores e parâmetros
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# Usar GridSearchCV para encontrar os melhores hiperparâmetros
# Definimos scoring='recall' para focar na detecção da classe minoritária
grid_search = GridSearchCV(
    estimator=xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', use_label_encoder=False, random_state=42),
    param_grid=param_grid,
    scoring='recall',
    cv=3, # Número de folds para validação cruzada
    verbose=1,
    n_jobs=-1 # Usar todos os núcleos da CPU
)

print("Iniciando Grid Search para ajuste de hiperparâmetros...")
grid_search.fit(X_train_res, y_train_res) # Treinar com dados balanceados

print("\nMelhores hiperparâmetros encontrados:")
print(grid_search.best_params_)

best_xgb_model = grid_search.best_estimator_

# Avaliar o modelo com os melhores hiperparâmetros
y_pred_tuned_xgb = best_xgb_model.predict(X_test)

print("\n--- Relatório de Classificação (XGBoost Otimizado) ---")
print(classification_report(y_test, y_pred_tuned_xgb))

print("\n--- Matriz de Confusão (XGBoost Otimizado) ---")
cm_tuned_xgb = confusion_matrix(y_test, y_pred_tuned_xgb)
print(cm_tuned_xgb)

# Visualizar a Matriz de Confusão do XGBoost Otimizado
fig, ax = plt.subplots(figsize=(6, 6))
cmd_tuned_xgb = ConfusionMatrixDisplay(confusion_matrix=cm_tuned_xgb, display_labels=['Não Anomalia', 'Anomalia'])
cmd_tuned_xgb.plot(cmap=plt.cm.Oranges, ax=ax)
plt.title('Matriz de Confusão (XGBoost Otimizado)')
plt.show()


### Explicabilidade do Modelo com SHAP

SHAP (SHapley Additive exPlanations) é uma técnica poderosa para explicar a saída de qualquer modelo de machine learning. Ele mostra como cada feature contribui para a previsão de uma instância individual e para o modelo como um todo.

In [ ]:
# Criar um explicador SHAP
explaner = shap.TreeExplainer(best_xgb_model)

# Calcular os valores SHAP para o conjunto de teste
shap_values = explaner.shap_values(X_test)

print("Sumário da Importância das Features (SHAP):")
# Plotar o sumário da importância das features
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title('Importância Global das Features (SHAP)')
plt.show()

print("\nDependência das Features (SHAP) - Exemplo da feature mais importante:")
# Plotar o gráfico de dependência para a feature mais importante (ajuste conforme a saída da importância)
# Substitua 'feature_mais_importante' pela feature de maior importância encontrada anteriormente
feature_mais_importante = feature_importances.iloc[0]['feature']
shap.dependence_plot(feature_mais_importante, shap_values, X_test, show=False)
plt.title(f'Dependência de {feature_mais_importante} nos Valores SHAP')
plt.show()

print("\nExplicando uma previsão individual (primeira instância do teste):")
# Explicar uma única previsão
shap.initjs()
shap.force_plot(explaner.expected_value, shap_values[0,:], X_test.iloc[0,:])


#### Conclusão:

Com o XGBoost e as técnicas de balanceamento, ajuste de hiperparâmetros e explicabilidade com SHAP, temos um modelo robusto e compreensível para a detecção de anomalias. A importância das variáveis e os gráficos SHAP nos ajudam a entender como o modelo chega às suas decisões, o que é crucial em cenários como a detecção de fraude, onde a interpretabilidade é tão importante quanto o desempenho.